# Do bills and spending rise together?

This notebook tests one claim against `monthly_summary.csv`:

> When household bills rise, total spending rises by at least as much.

Total spending is `money_out`. A rise is the change from one month to the next for the same account. The bill change has to be at least $200 so a one-dollar flicker is not counted.

Two kinds of rise are kept separate, because they are not the same event:

1. **Any jump.** The bill is at least $200 higher than last month. This includes a bill that was $0 last month and posts this month.
2. **Already paying.** The bill was already above $0, and then rose by at least $200.

`money_out` is the sum of household bills, insurance, loan payments, bank fees, cash withdrawals, card withdrawals, and other transfers. "Other spending" is `money_out` minus bills, insurance, and loan payments.


In [1]:
from collections import defaultdict
from pathlib import Path

candidates = []
for start in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(start / "data" / "monthly_summary.csv")
    candidates.append(start / "sheridan-datathon" / "data" / "monthly_summary.csv")
DATA = next(p for p in candidates if p.exists())

rows = defaultdict(list)
with DATA.open(newline="") as f:
    import csv
    reader = csv.DictReader(f)
    parts = [
        "household_bills", "insurance", "loan_payments", "bank_fees",
        "cash_withdrawals_genuine", "card_withdrawals", "transfers_out_other",
    ]
    mismatch = 0
    n_rows = 0
    for row in reader:
        n_rows += 1
        total = float(row["money_out"])
        summed = sum(float(row[c]) for c in parts)
        if abs(total - summed) > 1:
            mismatch += 1
        rows[row["account_id"]].append(row)

print(f"accounts: {len(rows):,}")
print(f"account-months: {n_rows:,}")
print(f"months where money_out is not the sum of its parts (tolerance $1): {mismatch}")


accounts: 4,500
account-months: 185,326
months where money_out is not the sum of its parts (tolerance $1): 0


In [2]:
def other_spend(month):
    return (
        float(month["money_out"])
        - float(month["household_bills"])
        - float(month["insurance"])
        - float(month["loan_payments"])
    )

def median(xs):
    xs = sorted(xs)
    n = len(xs)
    if n == 0:
        return None
    if n % 2:
        return xs[n // 2]
    return (xs[n // 2 - 1] + xs[n // 2]) / 2

jumps = []
for hist in rows.values():
    hist.sort(key=lambda r: r["month"])
    for prev, cur in zip(hist, hist[1:]):
        bill_prev = float(prev["household_bills"])
        bill_now = float(cur["household_bills"])
        bill_change = bill_now - bill_prev
        if bill_change < 200:
            continue
        spend_change = float(cur["money_out"]) - float(prev["money_out"])
        jumps.append({
            "bill_prev": bill_prev,
            "bill_change": bill_change,
            "spend_change": spend_change,
            "other_change": other_spend(cur) - other_spend(prev),
            "from_zero": bill_prev == 0,
        })

print(f"bill jumps of at least $200: {len(jumps):,}")
print(f"  of which the bill was $0 in the previous month: {sum(j['from_zero'] for j in jumps):,}")
print(f"  of which a bill was already being paid: {sum(not j['from_zero'] for j in jumps):,}")


bill jumps of at least $200: 4,511
  of which the bill was $0 in the previous month: 3,611
  of which a bill was already being paid: 900


In [3]:
def summarize(name, subset):
    n = len(subset)
    bill = [j["bill_change"] for j in subset]
    spend = [j["spend_change"] for j in subset]
    other = [j["other_change"] for j in subset]
    at_least = sum(s >= b for s, b in zip(spend, bill))
    faster = sum(s > b for s, b in zip(spend, bill))
    other_fell = sum(o < 0 for o in other)
    print(name)
    print(f"  months: {n:,}")
    print(f"  median bill increase: ${median(bill):,.1f}")
    print(f"  median total-spending increase: ${median(spend):,.1f}")
    print(f"  median other-spending change: ${median(other):,.1f}")
    print(f"  months where spending rose by at least the bill increase: {at_least:,} ({at_least / n:.1%})")
    print(f"  months where spending rose by more than the bill increase: {faster:,} ({faster / n:.1%})")
    print(f"  months where other spending fell: {other_fell:,} ({other_fell / n:.1%})")
    print()

summarize("All jumps of at least $200", jumps)
summarize("Bill was $0, then posted", [j for j in jumps if j["from_zero"]])
summarize("Bill was already above $0", [j for j in jumps if not j["from_zero"]])


All jumps of at least $200
  months: 4,511
  median bill increase: $2,122.5
  median total-spending increase: $2,844.7
  median other-spending change: $314.6
  months where spending rose by at least the bill increase: 2,682 (59.5%)
  months where spending rose by more than the bill increase: 2,612 (57.9%)
  months where other spending fell: 1,854 (41.1%)

Bill was $0, then posted
  months: 3,611
  median bill increase: $2,401.3
  median total-spending increase: $3,126.0
  median other-spending change: $500.0
  months where spending rose by at least the bill increase: 2,231 (61.8%)
  months where spending rose by more than the bill increase: 2,187 (60.6%)
  months where other spending fell: 1,405 (38.9%)

Bill was already above $0
  months: 900
  median bill increase: $1,069.8
  median total-spending increase: $1,088.6
  median other-spending change: $0.0
  months where spending rose by at least the bill increase: 451 (50.1%)
  months where spending rose by more than the bill increase: 

In [4]:
# Percent change is only defined when last month's bill and spending were above zero.
# That is the "already paying" group.
already = [j for j in jumps if not j["from_zero"]]
# Rebuild percent changes from the stored dollar changes and the previous bill.
# Previous spending is not stored, so recompute percent changes from the source rows.
pct_bill = []
pct_spend = []
for hist in rows.values():
    hist.sort(key=lambda r: r["month"])
    for prev, cur in zip(hist, hist[1:]):
        bill_prev = float(prev["household_bills"])
        spend_prev = float(prev["money_out"])
        bill_change = float(cur["household_bills"]) - bill_prev
        if bill_change < 200 or bill_prev <= 0 or spend_prev <= 0:
            continue
        pct_bill.append(bill_change / bill_prev)
        pct_spend.append((float(cur["money_out"]) - spend_prev) / spend_prev)

n = len(pct_bill)
at_least = sum(s >= b for s, b in zip(pct_spend, pct_bill))
print("Already-paying jumps, percent change")
print(f"  months: {n:,}")
print(f"  median percent change in the bill: {median(pct_bill):.1%}")
print(f"  median percent change in total spending: {median(pct_spend):.1%}")
print(f"  months where spending's percent change was at least the bill's: {at_least:,} ({at_least / n:.1%})")


Already-paying jumps, percent change
  months: 900
  median percent change in the bill: 25.0%
  median percent change in total spending: 13.8%
  months where spending's percent change was at least the bill's: 308 (34.2%)


## What the counts say

Across every jump of at least $200, the typical total-spending increase is larger than the typical bill increase, and spending rises by at least as much as the bill in about 60% of those months. Most of those jumps are a bill that was $0 and then posted. In that group, other spending rises as well.

When a bill was already being paid and then rose, the typical spending increase is about the same number of dollars as the bill increase, not a faster one. Other spending is flat. In percent terms the bill rises faster than total spending, and spending's percent change keeps up with the bill in only about a third of those months.

So the dollar claim holds for the mixed set of jumps, mostly because a new bill is itself new spending and other spending does not fall. It does not hold as "spending grows at a faster rate than bills" once the household was already paying the bill.
